# dictionary_methods
Dictionary methods are how Python code safely reads, mutates, merges, and restructures keyed state. In production systems they power config loading, JSON parsing, lookup tables, nested payload traversal, frequency counting, and derived indexing.

## 1. Problem First
Why does this matter in real systems?
- API payloads are usually dictionaries with optional fields.
- Config merging depends on careful update rules.
- One bad key assumption can crash a pipeline or silently lose data.

In [ ]:
config = {"host": "api.internal", "port": 8080}
print(config.get("host"))
print(config.get("debug", False))

## 2. Minimal Concept
Key dictionary tools and patterns:
- Methods: `get()`, `keys()`, `values()`, `items()`, `update()`, `pop()`, `popitem()`, `clear()`, `copy()`, `setdefault()`, `fromkeys()`
- Patterns: nested dictionaries, safe access, iteration, merging, unpacking, frequency counting, inversion, sorting, comprehension
- Related structures: `defaultdict`, `OrderedDict`, `Counter`

## 3. Mental Model
How Python actually behaves
Dictionaries are mutable hash-table-backed mappings. Read methods like `get()` do not mutate, while methods like `update()`, `pop()`, `clear()`, and `setdefault()` can. Views from `keys()`, `values()`, and `items()` are live, not frozen snapshots.

In [ ]:
settings = {"mode": "prod"}
items_view = settings.items()
print(list(items_view))
settings["debug"] = False
print(list(items_view))
print("mode" in settings, "timeout" in settings)

## 4. Internal Mechanics
Dictionary methods work against the same underlying object unless you explicitly copy. `dict[key]` can raise `KeyError`; `get()` gives safer fallback behavior. Merging and unpacking create or mutate mappings depending on which form you use.

In [ ]:
from collections import Counter, OrderedDict, defaultdict
import dis

def merge(base, override):
    base.update(override)
    return base

dis.dis(merge)
base = {"host": "a"}
override = {"port": 80}
print(id(base))
merge(base, override)
print(id(base), base)
print(dict.fromkeys(["a", "b"], 0))
print({**{"a": 1}, **{"b": 2}})
counts = Counter(["a", "b", "a"])
print(counts)
grouped = defaultdict(list)
grouped["errors"].append("timeout")
print(grouped)
ordered = OrderedDict([("x", 1), ("y", 2)])
print(ordered)

## 5. Edge Cases
Where it breaks:
- Direct key access raises `KeyError` for missing optional fields.
- `setdefault()` mutates the dictionary and can hide initialization side effects.
- Shallow copies still share nested mutable values.
- Inverting a dictionary fails if values are not unique or hashable.
- `update()` and unpacking can silently overwrite data.

In [ ]:
try:
    print({}["missing"])
except KeyError as exc:
    print(type(exc).__name__)
nested = {"meta": {"retries": 3}}
clone = nested.copy()
clone["meta"]["retries"] = 10
print(nested)
groups = {}
groups.setdefault("errors", []).append("disk")
print(groups)
source = {"a": 1, "b": 1}
inverted = {value: key for key, value in source.items()}
print(inverted)

## 6. Performance Thinking
- Average lookup, insert, and delete are O(1).
- Iterating `keys()`, `values()`, and `items()` is O(n).
- Frequency counting with dicts is usually O(n) overall.
- Sorting dictionary items is O(n log n) after materialization.
- Dictionaries are chosen to avoid repeated O(n) list scans.

## 7. Real Use Case
A config loader and payload normalizer often need nested dictionaries, safe access, merging, counting, sorting, inversion, and JSON-like traversal.

In [ ]:
payload = {
    "service": "api",
    "meta": {"region": "us", "retries": 2},
    "tags": ["prod", "critical"]
}
print(payload["meta"].get("region"))
for key, value in payload.items():
    print(key, value)
freq = {}
for char in "banana":
    freq[char] = freq.get(char, 0) + 1
print(freq)
sorted_items = sorted(freq.items(), key=lambda item: item[1], reverse=True)
print(sorted_items)

In [ ]:
data = {"a": 1, "b": 2}
print(list(data.keys()))
print(list(data.values()))
print(list(data.items()))
print(data.pop("a"))
print(data)
print(data.popitem())
copy_data = {"x": 1}.copy()
copy_data.clear()
print(copy_data)

## 8. Anti-Pattern
What beginners do wrong:
- Use direct indexing for optional values.
- Mutate original configs during merge when preservation was required.
- Use dict comprehensions or merges without thinking about overwrite rules.
- Treat `Counter`, `defaultdict`, and plain dict as interchangeable without reason.

In [ ]:
base = {"port": 8080, "debug": False}
override = {"debug": True}
base.update(override)
print(base)
print("update silently replaced the old value")
user_map = {user["id"]: user for user in [{"id": 1}, {"id": 1}]}
print(user_map)

## 9. Interview Signals
Questions FAANG asks:
- What is the difference between `get()` and direct indexing?
- Why are dict lookups usually O(1)?
- What does `setdefault()` do and why can it be risky?
- How would you count frequencies with a dict, `Counter`, or `defaultdict`?
- What can go wrong when inverting a dictionary?

## 10. Exercise (Non-trivial)
Design a config and payload processing layer using dictionaries. It should merge defaults, file config, and environment overrides, safely read optional keys, count error codes, invert selected mappings when safe, sort derived summaries, and explain when `Counter`, `defaultdict`, or `OrderedDict` improve the design.

In [ ]:
def build_runtime_config(defaults, file_config, env_config):
    config = defaults.copy()
    config.update(file_config)
    config.update(env_config)
    return config

# Task:
# 1. Add validation for required nested keys.
# 2. Add safe inversion only when values are unique.
# 3. Add frequency counting for repeated payload fields.
# 4. Explain where defaultdict, OrderedDict, and Counter would help.